<a href="https://colab.research.google.com/github/jlen0011-eng/Count_Apple/blob/main/Apple2210.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Mount Google Drive and Persist Dataset
Running the code below will prompt you to authorize Google Drive access. Once mounted, we can copy the file to your Drive for permanent storage.

In [1]:
from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Define paths
src_file = '/content/Dataset-20260930T233804Z-1-001.zip'
dest_dir = '/content/drive/MyDrive/Colab_Datasets/'
dest_file = os.path.join(dest_dir, 'Dataset-20260930T233804Z-1-001.zip')

# Create destination folder if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

# Copy file to Google Drive
if os.path.exists(src_file):
    shutil.copy(src_file, dest_file)
    print(f"Successfully saved {src_file} to Google Drive at: {dest_file}")
else:
    print(f"Source file {src_file} not found.")

Mounted at /content/drive
Successfully saved /content/Dataset-20260930T233804Z-1-001.zip to Google Drive at: /content/drive/MyDrive/Colab_Datasets/Dataset-20260930T233804Z-1-001.zip


### Extract Dataset and Split into Train, Test, and Validation Sets
We will extract the contents of the zip file, create the `summer_rgb` and `summer_d` directories, and then split the files into `train` (70%), `test` (15%), and `val` (15%) subfolders.

In [5]:
import subprocess
import os

# Let's check the local source file on Colab's temporary storage
local_zip = '/content/Dataset-20260930T233804Z-1-001.zip'
extract_path = '/content/extracted_dataset'

print("Checking local source file:")
if os.path.exists(local_zip):
    print(f"Local file size: {os.path.getsize(local_zip)} bytes")
    result = subprocess.run(['file', local_zip], capture_output=True, text=True)
    print("Local file type analysis:", result.stdout.strip())

    # Attempt to unzip the original local file directly
    os.makedirs(extract_path, exist_ok=True)
    print("\nAttempting extraction of local file...")
    res_unzip = subprocess.run(['unzip', '-q', local_zip, '-d', extract_path], capture_output=True, text=True)
    if res_unzip.returncode == 0:
        print("Successfully extracted local file!")
    else:
        print("Local unzip failed. Error:\n", res_unzip.stderr)
else:
    print("Local file /content/Dataset-20260930T233804Z-1-001.zip does not exist.")

Checking local source file:
Local file size: 745153004 bytes
Local file type analysis: /content/Dataset-20260930T233804Z-1-001.zip: Zip archive data, at least v2.0 to extract, compression method=deflate

Attempting extraction of local file...
Successfully extracted local file!


In [7]:
import os
import glob
import random
import shutil

base_dir = '/content/split_dataset'
extract_path = '/content/extracted_dataset'
subfolders = ['summer_rgb', 'summer_d']
splits = ['train', 'test', 'val']

# Create clean target directory structure
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
for sub in subfolders:
    for s in splits:
        os.makedirs(os.path.join(base_dir, sub, s), exist_ok=True)

# Recursively find all image files
all_files = glob.glob(os.path.join(extract_path, '**', '*'), recursive=True)
all_images = [f for f in all_files if os.path.isfile(f) and f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]

# Filter strictly for 'Summer' files
summer_images = [img for img in all_images if 'summer' in os.path.basename(img).lower()]

# Separate into rgb and depth (d) strictly for Summer
rgb_images = [img for img in summer_images if 'rgb' in os.path.basename(img).lower() or 'color' in os.path.basename(img).lower()]
d_images = [img for img in summer_images if '_d-' in os.path.basename(img).lower() or 'depth' in os.path.basename(img).lower()]

print(f"Total images: {len(all_images)}")
print(f"Summer-only images: {len(summer_images)}")
print(f" -> Summer RGB: {len(rgb_images)}")
print(f" -> Summer Depth (D): {len(d_images)}")

def split_and_copy(images, category):
    if not images:
        print(f"\nNo images to process for {category}")
        return

    random.seed(42)
    random.shuffle(images)

    n = len(images)
    # Proportions: 75% train, 15% test, 10% validation (sums to 100%)
    n_train = int(n * 0.75)
    n_test = int(n * 0.15)

    train_imgs = images[:n_train]
    test_imgs = images[n_train:n_train + n_test]
    val_imgs = images[n_train + n_test:]

    for img_set, split_name in zip([train_imgs, test_imgs, val_imgs], ['train', 'test', 'val']):
        dest_path = os.path.join(base_dir, category, split_name)
        for img in img_set:
            shutil.copy(img, os.path.join(dest_path, os.path.basename(img)))

    print(f"\n{category.upper()} split complete:")
    print(f" - Train (75%): {len(train_imgs)}")
    print(f" - Test (15%): {len(test_imgs)}")
    print(f" - Val (10%): {len(val_imgs)}")

split_and_copy(rgb_images, 'summer_rgb')
split_and_copy(d_images, 'summer_d')

Total images: 2176
Summer-only images: 1088
 -> Summer RGB: 544
 -> Summer Depth (D): 544

SUMMER_RGB split complete:
 - Train (75%): 408
 - Test (15%): 81
 - Val (10%): 55

SUMMER_D split complete:
 - Train (75%): 408
 - Test (15%): 81
 - Val (10%): 55
